<!-- torchleet:colab -->
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Fengyee/TorchLeet/blob/main/v3/gpu-systems/triton-fused-softmax/triton-fused-softmax.ipynb)

Check your work: `pip install -e python/` once from the repo root (on Colab: `!pip install -q "git+https://github.com/Fengyee/TorchLeet#subdirectory=python"`), then `from torchleet import check; check("triton-fused-softmax", ...)`


# Write a Fused Softmax Kernel in Triton

**Difficulty**: 🟣 Expert

**Companies**: NVIDIA, Meta, Google, xAI, Tesla

---

### Problem Statement

Standard softmax requires **3 separate passes** over data:
1. **Pass 1**: Compute the maximum value across the row (for numerical stability)
2. **Pass 2**: Subtract the max, exponentiate each element, and accumulate the sum
3. **Pass 3**: Divide each exponentiated value by the sum

Done as separate kernels, each pass reads the row from global memory (HBM) and the intermediate results are written back and re-read, so softmax is **memory-bandwidth bound**. A **fused kernel** reads each row from HBM once and writes the result once: when the whole row fits in on-chip memory (registers/SRAM), the max, the exponentials, the sum and the division all run on that on-chip copy.

When a row is too long to fit on-chip, the kernel has to stream it in blocks, and the **online softmax** (Milakov & Gimelshein, 2018) is what keeps that cheap. It finds the max and the normalizer in a *single* streaming pass by keeping a **running max** and a **running sum**, and rescales the sum by `exp(m_old − m_new)` whenever a new maximum appears. A second pass writes `exp(x − m) / l`. That is 2 reads + 1 write instead of 3 reads + 1 write, and the same rescaling trick is the core of FlashAttention.

Your task:
1. Implement the **online softmax algorithm** in pure PyTorch (`online_softmax_pytorch`), vectorized over rows and streaming over the columns in blocks
2. Write a **Triton fused softmax kernel** that processes one row per program instance (or provide the kernel code with a fallback if Triton is unavailable)
3. Validate both against `torch.softmax`

---

### Requirements

1. **Online Softmax** — Implement the single-pass algorithm: maintain a running max and running sum per row, rescaling the sum when a new max is found, then normalize in a second pass.
2. **Triton Kernel** — Write `fused_softmax_kernel` using `@triton.jit` that processes one row per program instance. Each program loads a row into SRAM, computes softmax in registers, and writes back.
3. **Launcher** — Write `fused_softmax(x)` that allocates output and launches the kernel with the right grid.
4. **Correctness** — All implementations must match `torch.softmax` within float32 tolerance.

---

### Constraints

- ✅ Triton kernel is optional: it runs on a CUDA GPU, or on CPU through Triton's interpreter (`pip install triton`, then set `TRITON_INTERPRET=1` *before* importing triton — slow, but it runs the same kernel code)
- ✅ The pure PyTorch online softmax must work on CPU
- ✅ Must handle row lengths that are not powers of two: `BLOCK_SIZE` is the next power of 2 ≥ `n_cols`, and loads/stores are masked (this one-block-per-row kernel assumes the row fits on-chip; much longer rows need the blocked online version)
- ❌ Do **not** use `torch.softmax` in your implementation

---

<details>
  <summary>💡 Hint</summary>

  **Online softmax recurrence:**
  ```
  For each element x_i in the row:
      new_max = max(running_max, x_i)
      running_sum = running_sum * exp(running_max - new_max) + exp(x_i - new_max)
      running_max = new_max
  
  Final output: exp(x_i - final_max) / final_sum
  ```
  
  For the Triton kernel:
  - Use `tl.program_id(0)` to get the row index
  - Use `tl.arange(0, BLOCK_SIZE)` to create column offsets
  - Use a mask for columns beyond the actual row length, and load those lanes as `-inf` so they contribute `exp(-inf) = 0` to the sum
  - `triton.next_power_of_2(n_cols)` gives the block size; the kernel assumes unit stride along a row, so make the input's columns contiguous
  - `tl.max`, `tl.exp`, and `tl.sum` operate on the loaded block

</details>

---

In [ ]:
import torch
import torch.nn.functional as F
import math
import os

# No GPU? Uncomment to run the Triton kernel on CPU with Triton's interpreter
# (must be set before triton is imported):
# os.environ["TRITON_INTERPRET"] = "1"

# Try to import Triton (may not be available on all systems)
TRITON_AVAILABLE = False
try:
    import triton
    import triton.language as tl
    TRITON_AVAILABLE = True
    print("Triton is available!")
except ImportError:
    print("Triton not available. Will use pure PyTorch implementation only.")

CUDA_AVAILABLE = torch.cuda.is_available()
print(f"CUDA available: {CUDA_AVAILABLE}")

# Triton kernels run on a GPU, or on CPU through Triton's interpreter
# (export TRITON_INTERPRET=1 *before* importing triton; slow, but handy for debugging).
TRITON_INTERPRET = os.environ.get("TRITON_INTERPRET") == "1"
RUN_TRITON = TRITON_AVAILABLE and (CUDA_AVAILABLE or TRITON_INTERPRET)
TRITON_DEVICE = "cuda" if CUDA_AVAILABLE else "cpu"

In [ ]:
# Test data
torch.manual_seed(42)

# 2D input: (batch, features)
x_small = torch.randn(4, 8)
x_medium = torch.randn(32, 128)
x_large_vals = torch.tensor([[1000.0, 2000.0, 3000.0, 4000.0]])

print("x_small shape:", x_small.shape)
print("x_medium shape:", x_medium.shape)
print("x_large_vals:", x_large_vals)

In [ ]:
def online_softmax_pytorch(x: torch.Tensor, block_size: int = 32) -> torch.Tensor:
    """
    Softmax over the last dim of a 2D tensor with the online algorithm.

    Pass 1 streams over the columns in blocks of `block_size`, keeping a running
    max m and running sum l for every row at once; when a block raises the max,
    the sum so far is rescaled by exp(m_old - m_new). Pass 2 writes exp(x - m) / l.
    block_size=1 is the textbook element-by-element recurrence.

    Args:
        x: Input tensor of shape (M, N)
    Returns:
        Softmax output of the same shape
    """
    assert x.dim() == 2, "Input must be 2D (M, N)"
    M, N = x.shape

    # TODO: Initialize a running max (-inf) and a running sum (0) for every row: shape (M, 1)

    # TODO: Pass 1 - sweep over the columns in blocks of block_size. For each block:
    #   new_max = max(running_max, max of the block)
    #   rescale the running sum to the new max, then add the block's sum of exp(x - new_max)
    #   running_max = new_max

    # TODO: Pass 2 - output = exp(x - running_max) / running_sum
    ...

In [ ]:
# Triton kernel (runs if Triton is installed, on a GPU or under TRITON_INTERPRET=1)
# If Triton is not available, this cell defines nothing.

if TRITON_AVAILABLE:
    @triton.jit
    def fused_softmax_kernel(
        output_ptr, input_ptr,
        input_row_stride, output_row_stride,
        n_cols,
        BLOCK_SIZE: tl.constexpr,
    ):
        """
        Fused softmax kernel: each program instance processes one row.

        Steps:
        1. Load the row into registers (with masking for padding)
        2. Compute max for numerical stability
        3. Subtract max and exponentiate
        4. Normalize by sum
        5. Store result back to HBM
        """
        # TODO: Which row does this program instance own? (tl.program_id)

        # TODO: Column offsets for one block (tl.arange) and a mask for the lanes
        #       past n_cols - BLOCK_SIZE is a power of 2 >= n_cols

        # TODO: Load the row (pointer = base + row * row_stride + offsets).
        #       What value must the masked lanes hold so they add nothing to the
        #       max or to the sum of exponentials?

        # TODO: Numerically stable softmax on the loaded row (tl.max, tl.exp, tl.sum)

        # TODO: Store the result into the output row, masked
        ...

    def fused_softmax(x: torch.Tensor) -> torch.Tensor:
        """
        Launch the Triton fused softmax kernel.

        Args:
            x: Input tensor of shape (M, N) (on CUDA, or CPU under TRITON_INTERPRET=1)
        Returns:
            Softmax output of the same shape
        """
        # TODO: The kernel walks along a row with unit stride - make sure x's columns are contiguous
        # TODO: Allocate the output tensor
        # TODO: Choose BLOCK_SIZE (one block must cover a whole row)
        # TODO: Launch the kernel with one program per row and return the output
        ...
else:
    print("Triton not available. Skipping kernel definition.")
    print("The fused_softmax function will not be available.")
    print("Focus on implementing online_softmax_pytorch above.")

In [ ]:
# Validation
print("=" * 60)
print("VALIDATING ONLINE SOFTMAX (PyTorch, CPU)")
print("=" * 60)

# Test 1: Small input
out_small = online_softmax_pytorch(x_small)
ref_small = torch.softmax(x_small, dim=-1)
print("\n--- Small Input Test ---")
print("Max absolute error:", (out_small - ref_small).abs().max().item())
assert torch.allclose(out_small, ref_small, atol=1e-6), "Small input test FAILED!"
print("PASSED")

# Test 2: Medium input
out_medium = online_softmax_pytorch(x_medium)
ref_medium = torch.softmax(x_medium, dim=-1)
print("\n--- Medium Input Test ---")
print("Max absolute error:", (out_medium - ref_medium).abs().max().item())
assert torch.allclose(out_medium, ref_medium, atol=1e-6), "Medium input test FAILED!"
print("PASSED")

# Test 3: Numerical stability with large values
out_large = online_softmax_pytorch(x_large_vals)
ref_large = torch.softmax(x_large_vals, dim=-1)
print("\n--- Numerical Stability Test ---")
print("Output:", out_large)
print("Reference:", ref_large)
assert not torch.isnan(out_large).any(), "NaN in output!"
assert not torch.isinf(out_large).any(), "Inf in output!"
assert torch.allclose(out_large, ref_large, atol=1e-6), "Large values test FAILED!"
print("PASSED")

# Test 4: Sum-to-1 property
print("\n--- Sum-to-1 Test ---")
sums = out_small.sum(dim=-1)
print("Row sums:", sums)
assert torch.allclose(sums, torch.ones_like(sums)), "Rows do not sum to 1!"
print("PASSED")

print("\n" + "=" * 60)
print("VALIDATING TRITON FUSED SOFTMAX (GPU, or CPU via TRITON_INTERPRET=1)")
print("=" * 60)

if RUN_TRITON:
    # Includes a row length that is not a power of 2 and very large values
    for name, x_t in [("medium", x_medium), ("ragged 1000 cols", torch.randn(16, 1000) * 5),
                      ("large values", x_large_vals)]:
        x_t = x_t.to(TRITON_DEVICE)
        out_triton = fused_softmax(x_t)
        ref_t = torch.softmax(x_t, dim=-1)
        print(f"\n{name}: max absolute error {(out_triton - ref_t).abs().max().item():.2e}")
        assert torch.allclose(out_triton, ref_t, atol=1e-6), f"Triton softmax FAILED on {name}!"
    print("Triton fused softmax PASSED!")
else:
    print("\nSkipping Triton validation (needs Triton plus CUDA or TRITON_INTERPRET=1).")
    print("Online softmax PyTorch implementation validated successfully.")

print("\nAll available tests passed!")

In [ ]:
# Auto-grade (one-time setup from the repo root: pip install -e python/)
from torchleet import check
# To also verify your Triton kernel (GPU, or CPU with TRITON_INTERPRET=1), add
# fused_softmax_triton=fused_softmax to the call.
check("triton-fused-softmax", online_softmax_pytorch=online_softmax_pytorch)